In [26]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal
from scipy.stats import mannwhitneyu



PreRequisites Before Hypotheses Testing

Checking for Blocked Implementations or Implementations which could not compile

In [27]:
import pandas as pd
prep_status = pd.read_excel("AnaylsisforKruskalWallis.xlsx")
prep_frames = []
for prep_task in range(1, 10):
    prep_part = pd.read_excel(
        f"Task{prep_task}_Error_Counts_and_Categories.xlsx",
        sheet_name="Error Counts"
    )
    prep_part = prep_part[
        pd.to_numeric(
            prep_part["Run"], errors="coerce"
        ).notna()
    ].copy()
   
    prep_part["Task"] = prep_task
    prep_frames.append(prep_part)
prep_counts = pd.concat(prep_frames, ignore_index=True)



display(prep_counts.head())

,Run,Test source,Implementation source,Implementation errors,Test errors,Task
0,1,Deepseek,Deepseek,0,6,1
1,1,Deepseek,Gemma,0,6,1
2,1,Deepseek,Mistral,0,6,1
3,1,Deepseek,OpenAI,0,6,1
4,1,Deepseek,Amazon,0,6,1


In [28]:
prep_status_work = prep_status.copy()

for prep_col in ["Test source", "Implementation source"]:
    prep_status_work[prep_col] = (
        prep_status_work[prep_col].str.strip().str.lower()
    )

prep_status_work["Execution status"] = (
    prep_status_work["Execution status"].str.strip().str.lower()
)

prep_status_work["Is blocked"] = (
    prep_status_work["Execution status"] == "blocked"
)

prep_impl_summary = (
    prep_status_work
    .groupby(["Task", "Run", "Implementation source"])
    .agg(
        Comparisons=("Test source", "size"),
        Sources=("Test source", "nunique"),
        Blocked=("Is blocked", "sum")
    )
    .reset_index()
)

prep_impl_failures = prep_impl_summary[
    (prep_impl_summary["Comparisons"] == 6)
    & (prep_impl_summary["Sources"] == 6)
    & (prep_impl_summary["Blocked"] == 6)
].copy()

print("Implementation-loading failures:", len(prep_impl_failures))

display(
    prep_impl_failures[
        ["Task", "Run", "Implementation source", "Blocked"]
    ].reset_index(drop=True)
)

Implementation-loading failures: 22


,Task,Run,Implementation source,Blocked
0,1,4,gemma,6
1,1,5,gemma,6
2,1,6,gemma,6
3,1,9,gemma,6
4,2,3,gemma,6
5,2,9,gemma,6
6,4,6,mistral,6
7,4,8,gemma,6
8,4,9,gemma,6
9,5,9,gemma,6


In [29]:
prep_generated_impl = prep_impl_summary[
    prep_impl_summary["Implementation source"] != "manual"
].copy()
prep_manual_impl = prep_impl_summary[
    prep_impl_summary["Implementation source"] == "manual"
].drop_duplicates(subset=["Task"]).copy()
prep_impl_instances = pd.concat(
    [prep_generated_impl, prep_manual_impl],
    ignore_index=True
)
prep_impl_instances["Complexity"] = pd.cut(
    prep_impl_instances["Task"],
    bins=[0, 4, 7, 9],
    labels=["Simple", "Medium", "Complex"]
)
prep_impl_instances["Loaded"] = (
    prep_impl_instances["Blocked"]
    < prep_impl_instances["Comparisons"]
)
prep_impl_rates = (
    prep_impl_instances
    .groupby(
        ["Implementation source", "Complexity"],
        observed=True
    )
    .agg(
        Created=("Loaded", "size"),
        Loaded=("Loaded", "sum")
    )
    .reset_index()
)
prep_impl_rates["Result"] = prep_impl_rates.apply(
    lambda row: (
        f"{int(row['Loaded'])}/{int(row['Created'])} "
        f"({100 * row['Loaded'] / row['Created']:.2f}%)"
    ),
    axis=1
)
prep_impl_table = prep_impl_rates.pivot(
    index="Implementation source",
    columns="Complexity",
    values="Result"
).reindex(
    ["deepseek", "gemma", "mistral", "openai", "amazon", "manual"]
)

display(prep_impl_table)
 

Complexity,Simple,Medium,Complex
Implementation source,,,
deepseek,40/40 (100.00%),30/30 (100.00%),19/20 (95.00%)
gemma,32/40 (80.00%),27/30 (90.00%),17/20 (85.00%)
mistral,39/40 (97.50%),30/30 (100.00%),20/20 (100.00%)
openai,40/40 (100.00%),30/30 (100.00%),20/20 (100.00%)
amazon,40/40 (100.00%),24/30 (80.00%),20/20 (100.00%)
manual,4/4 (100.00%),3/3 (100.00%),2/2 (100.00%)


In [30]:
prep_eligible_status = prep_status_work.merge(
    prep_impl_failures[
        ["Task", "Run", "Implementation source"]
    ],
    on=["Task", "Run", "Implementation source"],
    how="left",
    indicator=True
)
prep_eligible_status = prep_eligible_status[
    prep_eligible_status["_merge"] == "left_only"
].drop(columns="_merge").copy()

prep_test_summary = (
    prep_eligible_status
    .groupby(["Task", "Run", "Test source"])
    .agg(
        Comparisons=("Implementation source", "size"),
        Blocked=("Is blocked", "sum")
    )
    .reset_index()
)
prep_test_failures = prep_test_summary[
    (prep_test_summary["Blocked"] > 0)
    & (
        prep_test_summary["Blocked"]
        == prep_test_summary["Comparisons"]
    )
].copy()
print("Test-suite loading failures:", len(prep_test_failures))

display(
    prep_test_failures[
        ["Task", "Run", "Test source", "Comparisons", "Blocked"]
    ].reset_index(drop=True)
)

Test-suite loading failures: 7


,Task,Run,Test source,Comparisons,Blocked
0,2,4,gemma,6,6
1,5,2,deepseek,6,6
2,6,1,gemma,6,6
3,8,2,amazon,6,6
4,8,4,amazon,6,6
5,8,9,amazon,6,6
6,8,10,deepseek,6,6


In [31]:
prep_test_attempts = prep_eligible_status.copy()

prep_test_attempts["Complexity"] = pd.cut(
    prep_test_attempts["Task"],
    bins=[0, 4, 7, 9],
    labels=["Simple", "Medium", "Complex"]
)
prep_test_attempts["Loaded"] = (
    ~prep_test_attempts["Is blocked"]
)
prep_generated_tests = prep_test_attempts[
    prep_test_attempts["Test source"] != "manual"
].copy()

prep_manual_tests = (
    prep_test_attempts[
        prep_test_attempts["Test source"] == "manual"
    ]
    .groupby(
        ["Task", "Test source", "Complexity"],
        observed=True,
        as_index=False
    )["Loaded"]
    .any()
)
prep_test_records = pd.concat(
    [
        prep_generated_tests[
            ["Test source", "Complexity", "Loaded"]
        ],
        prep_manual_tests[
            ["Test source", "Complexity", "Loaded"]
        ]
    ],
    ignore_index=True
)
prep_test_rates = (
    prep_test_records
    .groupby(["Test source", "Complexity"], observed=True)
    .agg(
        Considered=("Loaded", "size"),
        Loaded=("Loaded", "sum")
    )
    .reset_index()
)
prep_test_rates["Result"] = prep_test_rates.apply(
    lambda row: (
        f"{int(row['Loaded'])}/{int(row['Considered'])} "
        f"({100 * row['Loaded'] / row['Considered']:.2f}%)"
    ),
    axis=1
)
prep_test_table = prep_test_rates.pivot(
    index="Test source",
    columns="Complexity",
    values="Result"
).reindex(
    ["deepseek", "gemma", "mistral", "openai", "amazon", "manual"]
)

display(prep_test_table)

Complexity,Simple,Medium,Complex
Test source,,,
deepseek,231/231 (100.00%),165/171 (96.49%),110/116 (94.83%)
gemma,225/231 (97.40%),165/171 (96.49%),116/116 (100.00%)
mistral,231/231 (100.00%),171/171 (100.00%),116/116 (100.00%)
openai,231/231 (100.00%),171/171 (100.00%),116/116 (100.00%)
amazon,231/231 (100.00%),171/171 (100.00%),98/116 (84.48%)
manual,4/4 (100.00%),3/3 (100.00%),2/2 (100.00%)


In [32]:
prep_analysis = prep_counts.copy()
for prep_col in ["Test source", "Implementation source"]:
    prep_analysis[prep_col] = (
        prep_analysis[prep_col].str.strip().str.lower()
    )
for prep_col in ["Implementation errors", "Test errors"]:
    prep_analysis[prep_col] = pd.to_numeric(
        prep_analysis[prep_col],
        errors="raise"
    )
prep_analysis = prep_analysis.merge(
    prep_impl_failures[
        ["Task", "Run", "Implementation source"]
    ],
    on=["Task", "Run", "Implementation source"],
    how="left",
    indicator=True
)
prep_analysis = (
    prep_analysis[
        prep_analysis["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)
print("Original comparisons:", len(prep_counts))
print("Removed comparisons:", len(prep_counts) - len(prep_analysis))
print("Retained comparisons:", len(prep_analysis))

Original comparisons: 3240
Removed comparisons: 132
Retained comparisons: 3108


Hypothesis Testing

In [33]:
df=pd.read_excel("AnaylsisforKruskalWallis.xlsx")
df.head()
print(len(df))

3240


In [34]:
executed=prep_analysis.rename(columns={"Test errors":"Test-error count"}).copy()
executed["Test source"]=executed["Test source"].str.capitalize().replace({"Openai":"OpenAI"})
models=["Deepseek","Gemma","Mistral","OpenAI","Amazon"]
groups=[executed.loc[executed["Test source"]==model,"Test-error count"].to_numpy(dtype=float) for model in models]
summary=executed[executed["Test source"].isin(models)].groupby("Test source")["Test-error count"].agg(["count","sum","mean","median"])
display(summary)

,count,sum,mean,median
Test source,,,,
Amazon,518,514,0.992278,0.0
Deepseek,518,328,0.633205,0.0
Gemma,518,432,0.833977,0.0
Mistral,518,861,1.662162,1.0
OpenAI,518,202,0.389961,0.0


We use Kruskal Wallis Test since it is a non parametic statistical test which doesnt need their data to be in a normal distribution. It is used to determine if there is a valid significant difference between the LLMs used  

In [35]:
H,p=kruskal(*groups)
print("H-statistic:",H)
print("p-value:",p)
if p<0.05:
    print("Reject H0: Significant difference between LLMs")
else:
    print("Fail to reject H0: No significant difference established")

H-statistic: 428.25547613031875
p-value: 2.1787196321063057e-91
Reject H0: Significant difference between LLMs


Since p value is less than 0.05(significance value) we reject the Null Hypothesis. That means there is significant difference between LLMS

To narrow down, we try to test which of the combinations or pairs at a time using ranks. For that we use Mann Whitney U test to find where there is a difference.

In [36]:
from itertools import combinations
from statsmodels.stats.multitest import multipletests

In [37]:
results=[]
for a,b in combinations(models,2):
    x=executed.loc[executed["Test source"]==a,"Test-error count"].to_numpy(dtype=float)
    y=executed.loc[executed["Test source"]==b,"Test-error count"].to_numpy(dtype=float)
    U,p=mannwhitneyu(x,y,alternative="two-sided")
    results.append([a,b,U,p])
pairwise=pd.DataFrame(results,columns=["LLM 1","LLM 2","U-statistic","p-value"])
pairwise["Holm p-value"]=multipletests(pairwise["p-value"],method="holm")[1]
pairwise["Significant"]=pairwise["Holm p-value"]<0.05

display(pairwise)


,LLM 1,LLM 2,U-statistic,p-value,Holm p-value,Significant
0,Deepseek,Gemma,137845.0,3.345008e-01,6.690017e-01,False
1,Deepseek,Mistral,62621.0,5.306419e-56,4.775777e-55,True
2,Deepseek,OpenAI,132094.0,6.028807e-01,6.690017e-01,False
3,Deepseek,Amazon,117534.5,4.954848e-05,2.477424e-04,True
4,Gemma,Mistral,63882.0,1.359564e-54,1.087651e-53,True
5,Gemma,OpenAI,126682.0,5.551864e-02,1.665559e-01,False
6,Gemma,Amazon,115834.5,5.613657e-06,3.368194e-05,True
7,Mistral,OpenAI,213271.0,4.688394e-68,4.688394e-67,True
8,Mistral,Amazon,185276.0,1.233338e-28,8.633366e-28,True
9,OpenAI,Amazon,118839.0,2.240964e-04,8.963856e-04,True


Holm correction was used to check if any false positives were presented since they might have a chance in producing a false poisitve.

In [38]:
from statsmodels.stats.multitest import multipletests
pairwise["Holm p-value"]=multipletests(pairwise["p-value"],alpha=0.05,method="holm")[1]
pairwise["Significant"]=pairwise["Holm p-value"]<0.05
display(pairwise)

,LLM 1,LLM 2,U-statistic,p-value,Holm p-value,Significant
0,Deepseek,Gemma,137845.0,3.345008e-01,6.690017e-01,False
1,Deepseek,Mistral,62621.0,5.306419e-56,4.775777e-55,True
2,Deepseek,OpenAI,132094.0,6.028807e-01,6.690017e-01,False
3,Deepseek,Amazon,117534.5,4.954848e-05,2.477424e-04,True
4,Gemma,Mistral,63882.0,1.359564e-54,1.087651e-53,True
5,Gemma,OpenAI,126682.0,5.551864e-02,1.665559e-01,False
6,Gemma,Amazon,115834.5,5.613657e-06,3.368194e-05,True
7,Mistral,OpenAI,213271.0,4.688394e-68,4.688394e-67,True
8,Mistral,Amazon,185276.0,1.233338e-28,8.633366e-28,True
9,OpenAI,Amazon,118839.0,2.240964e-04,8.963856e-04,True


In [39]:
n=sum(len(g) for g in groups)
k=len(groups)
eta2=(H-k+1)/(n-k)
print("Total observations:",n)
print("Effect size:",round(eta2,4))


Total observations: 2590
Effect size: 0.1641


Since the effect size is too large , this tells that therre is substantial differences in test error count 

LLMs VS Manual


In [40]:
manual=executed.loc[executed["Test source"]=="Manual","Test-error count"].to_numpy(dtype=float)
llm=executed.loc[executed["Test source"].isin(models),"Test-error count"].to_numpy(dtype=float)
print("Manual observations:",len(manual))
print("LLM observations:",len(llm))
print("Manual mean:",manual.mean())
print("LLM mean:",llm.mean())
U,p_manual=mannwhitneyu(manual,llm,alternative="two-sided")
print("U-statistic:",U)
print("p-value:",p_manual)
if p_manual<0.05:
    print("Reject H0: Significant difference between Manual and LLM tests")
else:
    print("Fail to reject H0: No significant difference established")


Manual observations: 518
LLM observations: 2590
Manual mean: 0.8378378378378378
LLM mean: 0.9023166023166023
U-statistic: 673047.5
p-value: 0.8929670671967916
Fail to reject H0: No significant difference established


p=0.8693 is greater than 0.05, meaning we do not have sufficient statistical evidence of a difference between the two groups.

In [41]:
cles=1-(U/(len(manual)*len(llm)))
print("CLES:",round(cles,4))


CLES: 0.4983


In [42]:
rng=np.random.default_rng(42)
cles_values=[]
for i in range(1000):
    m=rng.choice(manual,size=len(manual),replace=True)
    l=rng.choice(llm,size=len(llm),replace=True)
    u,_=mannwhitneyu(m,l,alternative="two-sided")
    cles_values.append(1-u/(len(m)*len(l)))
    

In [43]:

lower,upper=np.percentile(cles_values,[2.5,97.5])
print("CLES:",round(cles,4))
print("95% CI:",round(lower,4),round(upper,4))


CLES: 0.4983
95% CI: 0.4751 0.5225


H2

In [44]:
h2_parts=[]

for task in range(1,10):
    part=pd.read_excel(
        f"Task{task}_Error_Counts_and_Categories.xlsx",
        sheet_name="Error Counts"
    )
    part=part[pd.to_numeric(part["Run"],errors="coerce").notna()].copy()
    part["Task"]=task
    h2_parts.append(part)

h2=pd.concat(h2_parts,ignore_index=True)

for col in ["Test source","Implementation source"]:
    h2[col]=h2[col].str.strip().str.lower()

h2=h2[h2["Implementation source"]!="manual"].copy()

h2["Group"]="Other"
h2.loc[h2["Test source"]==h2["Implementation source"],"Group"]="Same"
h2.loc[h2["Test source"]=="manual","Group"]="Manual"

print("H2 comparisons:",len(h2))

H2 comparisons: 2700


In [45]:
blocked=[
    (1,"gemma",[4,5,6,9]),
    (2,"gemma",[3,9]),
    (4,"gemma",[8,9]),
    (4,"mistral",[6]),
    (5,"gemma",[9]),
    (6,"amazon",[4,5,6,7,8,9]),
    (7,"gemma",[7,10]),
    (8,"gemma",[6]),
    (9,"gemma",[3,9]),
    (9,"deepseek",[4])
]

h2_filtered=h2.copy()
for task,model,runs in blocked:
    h2_filtered=h2_filtered[
        ~((h2_filtered["Task"]==task)&
          (h2_filtered["Implementation source"]==model)&
          (h2_filtered["Run"].isin(runs)))
    ]

print("Comparisons before:",len(h2))
print("Comparisons removed:",len(h2)-len(h2_filtered))
print("Comparisons remaining:",len(h2_filtered))


Comparisons before: 2700
Comparisons removed: 132
Comparisons remaining: 2568


In [46]:
from scipy.stats import kruskal

h2_numeric=h2_filtered.copy()
h2_numeric["Implementation errors"]=pd.to_numeric(
    h2_numeric["Implementation errors"],errors="coerce"
)

if h2_numeric["Implementation errors"].isna().any():
    raise ValueError("Check the missing counts before continuing")

manual=h2_numeric.loc[h2_numeric["Group"]=="Manual","Implementation errors"]
same=h2_numeric.loc[h2_numeric["Group"]=="Same","Implementation errors"]
other=h2_numeric.loc[h2_numeric["Group"]=="Other","Implementation errors"]

print("Manual:",len(manual))
print("Same:",len(same))
print("Other:",len(other))

H_h2,p_h2=kruskal(manual,same,other)

print("H-statistic:",H_h2)
print("p-value:",p_h2)

Manual: 428
Same: 428
Other: 1712
H-statistic: 5.366704440575373
p-value: 0.06833370016296834


We fail to reject null hypothesis. that means we did not find a statistical significant difference between the three groups. 

In [47]:
n_h2=len(h2_numeric)
k_h2=3

eta2_h2=(H_h2-k_h2+1)/(n_h2-k_h2)

print("Effect size:",eta2_h2)

Effect size: 0.0013125553374562858


difference between the three groups was small

H3


In [48]:
h3 = h2_numeric.copy()

h3_models = ["deepseek", "gemma", "mistral", "openai", "amazon"]

assert len(h3) == 2568
assert set(h3["Implementation source"]) == set(h3_models)
assert h3["Implementation errors"].notna().all()

display(
    h3.groupby("Implementation source")["Implementation errors"]
      .agg(["count", "mean", "median"])
      .reindex(h3_models)
)

h3_groups = [
    h3.loc[
        h3["Implementation source"] == model,
        "Implementation errors"
    ].to_numpy(dtype=float)
    for model in h3_models
]

H_h3, p_h3 = kruskal(*h3_groups)
eta2_h3 = (H_h3 - len(h3_models) + 1) / (len(h3) - len(h3_models))

print("H-statistic:", H_h3)
print("p-value:", p_h3)
print("Effect size:", eta2_h3)

if p_h3 < 0.05:
    print("Reject H3 null: significant differences between implementation LLMs.")
else:
    print("Fail to reject H3 null: no significant difference established.")

,count,mean,median
Implementation source,,,
deepseek,534,0.359551,0.0
gemma,456,0.484649,0.0
mistral,534,1.123596,0.0
openai,540,0.400000,0.0
amazon,504,1.990079,1.0


H-statistic: 282.01714935568793
p-value: 8.18594240810676e-60
Effect size: 0.10847333178138428
Reject H3 null: significant differences between implementation LLMs.


H3

In [49]:
h3 = h2_numeric.copy()

print("Total comparisons:", len(h3))

h3_summary = (
    h3.groupby("Implementation source")["Implementation errors"]
      .agg(["count", "mean", "median"])
)

display(h3_summary)

Total comparisons: 2568


,count,mean,median
Implementation source,,,
amazon,504,1.990079,1.0
deepseek,534,0.359551,0.0
gemma,456,0.484649,0.0
mistral,534,1.123596,0.0
openai,540,0.400000,0.0


In [50]:
h3_models = ["deepseek", "gemma", "mistral", "openai", "amazon"]
h3_groups = [
    h3.loc[
        h3["Implementation source"] == model,
        "Implementation errors"
    ].to_numpy(dtype=float)
    for model in h3_models
]

H_h3, p_h3 = kruskal(*h3_groups)

print("H-statistic:", H_h3)
print("p-value:", p_h3)

if p_h3 < 0.05:
    print("Significant: follow up with pairwise comparisons.")
else:
    print("Not significant: no pairwise comparisons needed.")

H-statistic: 282.01714935568793
p-value: 8.18594240810676e-60
Significant: follow up with pairwise comparisons.


We therefore reject null hypothesis of H3 that is there is evidence that a statistical significant difference is present just because LLM changes for generating implementations.

In [51]:
h3_results = []

for (a, x), (b, y) in combinations(zip(h3_models, h3_groups), 2):
    u, p = mannwhitneyu(x, y, alternative="two-sided")
    h3_results.append([a, b, u, p])

h3_pairwise = pd.DataFrame(
    h3_results,
    columns=["LLM 1", "LLM 2", "U-statistic", "Raw p-value"]
)

h3_pairwise["Holm p-value"] = multipletests(
    h3_pairwise["Raw p-value"], method="holm"
)[1]

h3_pairwise["Significant"] = h3_pairwise["Holm p-value"] < 0.05

display(h3_pairwise)

,LLM 1,LLM 2,U-statistic,Raw p-value,Holm p-value,Significant
0,deepseek,gemma,126154.5,1.093150e-01,3.279450e-01,False
1,deepseek,mistral,120646.5,5.424137e-09,2.169655e-08,True
2,deepseek,openai,146932.0,3.904517e-01,7.809035e-01,False
3,deepseek,amazon,85900.5,3.120621e-33,2.496497e-32,True
4,gemma,mistral,99833.5,1.823672e-11,1.276570e-10,True
5,gemma,openai,120892.5,4.048683e-01,7.809035e-01,False
6,gemma,amazon,71532.0,7.279483e-34,6.551535e-33,True
7,mistral,openai,168585.5,5.463086e-11,3.277852e-10,True
8,mistral,amazon,107646.5,3.514079e-10,1.757039e-09,True
9,openai,amazon,85162.5,1.955172e-36,1.955172e-35,True


holm p value < 0.05 = Significant ( True)
holm p value > 0.05 = Not Significant ( True)

In [52]:
n_h3 = len(h3)
k_h3 = len(h3_groups)
eta2_h3 = (H_h3 - k_h3 + 1) / (n_h3 - k_h3)
print("H3 overall effect size:", round(eta2_h3, 4))

H3 overall effect size: 0.1085


In [53]:
h3_data = dict(zip(h3_models, h3_groups))
h3_cles = []
for _, row in h3_pairwise.iterrows():
    a = row["LLM 1"]
    b = row["LLM 2"]
    n_a = len(h3_data[a])
    n_b = len(h3_data[b])

    cles = row["U-statistic"] / (n_a * n_b)
    h3_cles.append(cles)
h3_pairwise["CLES: LLM 1 higher"] = h3_cles

display(h3_pairwise)

,LLM 1,LLM 2,U-statistic,Raw p-value,Holm p-value,Significant,CLES: LLM 1 higher
0,deepseek,gemma,126154.5,1.093150e-01,3.279450e-01,False,0.518080
1,deepseek,mistral,120646.5,5.424137e-09,2.169655e-08,True,0.423089
2,deepseek,openai,146932.0,3.904517e-01,7.809035e-01,False,0.509544
3,deepseek,amazon,85900.5,3.120621e-33,2.496497e-32,True,0.319171
4,gemma,mistral,99833.5,1.823672e-11,1.276570e-10,True,0.409987
5,gemma,openai,120892.5,4.048683e-01,7.809035e-01,False,0.490954
6,gemma,amazon,71532.0,7.279483e-34,6.551535e-33,True,0.311247
7,mistral,openai,168585.5,5.463086e-11,3.277852e-10,True,0.584636
8,mistral,amazon,107646.5,3.514079e-10,1.757039e-09,True,0.399971
9,openai,amazon,85162.5,1.955172e-36,1.955172e-35,True,0.312913


In [54]:
h3_rng=np.random.default_rng(42)
h3_lower=[]
h3_upper=[]
for _,row in h3_pairwise.iterrows():
    a=h3_data[row["LLM 1"]]
    b=h3_data[row["LLM 2"]]
    h3_boot=[]
    for i in range(1000):
        x=h3_rng.choice(a,size=len(a),replace=True)
        y=h3_rng.choice(b,size=len(b),replace=True)
        h3_u=mannwhitneyu(x,y,alternative="two-sided",method="asymptotic").statistic
        h3_boot.append(h3_u/(len(x)*len(y)))
    lower,upper=np.percentile(h3_boot,[2.5,97.5])
    h3_lower.append(lower)
    h3_upper.append(upper)
h3_pairwise["CLES CI lower"]=h3_lower
h3_pairwise["CLES CI upper"]=h3_upper
display(h3_pairwise[[
    "LLM 1","LLM 2","CLES: LLM 1 higher",
    "CLES CI lower","CLES CI upper"
]].round(4))

,LLM 1,LLM 2,CLES: LLM 1 higher,CLES CI lower,CLES CI upper
0,deepseek,gemma,0.5181,0.4958,0.5405
1,deepseek,mistral,0.4231,0.3974,0.4468
2,deepseek,openai,0.5095,0.4889,0.5296
3,deepseek,amazon,0.3192,0.2914,0.3466
4,gemma,mistral,0.4100,0.3858,0.4347
5,gemma,openai,0.4910,0.4694,0.5145
6,gemma,amazon,0.3112,0.2843,0.3390
7,mistral,openai,0.5846,0.5596,0.6092
8,mistral,amazon,0.4000,0.3691,0.4326
9,openai,amazon,0.3129,0.2880,0.3385
